<a href="https://colab.research.google.com/github/HeyMahdy/Multi-Agent-Cell-Annotation/blob/main/Muilti_Agent_Cell_Annotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scgpt scanpy gdown -q

In [ ]:
!pip install -q langchain langgraph langchain-openai langchain-community langsmith biopython pandas numpy tqdm

**scGpt Config**

In [ ]:
import sys

for key in list(sys.modules.keys()):
    if "scgpt" in key or "torchtext" in key:
        del sys.modules[key]

print("Cache cleared:", [k for k in sys.modules if "torchtext" in k or "scgpt" in k])

# ── STEP 2: Build and inject the fake torchtext ──
from types import ModuleType

class _Vocab:
    def __init__(self, vocab=None):
        self._stoi = dict(vocab) if isinstance(vocab, dict) else {}
        self._itos = list(self._stoi.keys())
        self._default_index = None

    def __len__(self):              return len(self._stoi)
    def __contains__(self, token):  return token in self._stoi
    def get_stoi(self):             return self._stoi
    def get_itos(self):             return self._itos

    def __getitem__(self, token):
        return self._stoi.get(
            token,
            self._default_index if self._default_index is not None else 0
        )

    def __call__(self, tokens):
        if isinstance(tokens, list):
            return [self._stoi.get(t, self._default_index or 0) for t in tokens]
        return self._stoi.get(tokens, self._default_index or 0)

    def set_default_index(self, idx):   self._default_index = idx
    def get_default_index(self):        return self._default_index

    def append_token(self, token):
        if token not in self._stoi:
            self._stoi[token] = len(self._itos)
            self._itos.append(token)

    def insert_token(self, token, index):
        if token not in self._stoi:
            self._itos.insert(index, token)
            self._stoi = {t: i for i, t in enumerate(self._itos)}


class _VocabBuilt(_Vocab):
    def __init__(self, ordered_dict, min_freq=1):
        super().__init__()
        for token, freq in ordered_dict.items():
            if freq >= min_freq:
                self._stoi[token] = len(self._itos)
                self._itos.append(token)
        self.vocab = self._stoi


def _vocab_builder(ordered_dict, min_freq=1, **kwargs):
    return _VocabBuilt(ordered_dict, min_freq)


_torchtext       = ModuleType("torchtext")
_torchtext_vocab = ModuleType("torchtext.vocab")
_torchtext_vocab.Vocab = _Vocab
_torchtext_vocab.vocab = _vocab_builder
_torchtext.vocab = _torchtext_vocab

sys.modules["torchtext"]       = _torchtext
sys.modules["torchtext.vocab"] = _torchtext_vocab


In [ ]:
import os
os.makedirs("/kaggle/working/scgpt_ckpt", exist_ok=True)

!gdown --folder "https://drive.google.com/drive/folders/1oWh_-ZRdhtoGQ2Fw24HP41FgLoomVo-y" \
        -O /kaggle/working/scgpt_ckpt

print("Files downloaded:", os.listdir("/kaggle/working/scgpt_ckpt"))
print("✅ Cell 4 done — model downloaded")